# XAUUSD v2 — Volatilidad + Bayes: Fourier → GARCH → Bias bayesiano → Monte Carlo Markov

**Qué agrega esta versión sobre el pipeline anterior:**
1. **Arranque bayesiano del Monte Carlo**: en vez de asumir un único régimen actual, los 10,000
   caminos arrancan repartidos según la *probabilidad posterior* de cada régimen (el HMM ya la calcula).
2. **Motor de volatilidad**: GARCH(1,1) pronostica cuánto se moverá el oro hora a hora; un HMM
   aparte clasifica el mercado en CALMA / NORMAL / TORMENTA; y el perfil intradía hace que las
   bandas se ensanchen en Londres–NY y se estrechen en Asia.
3. **Bias bayesiano**: combina régimen de precio, régimen de volatilidad, momentum y sesión en una
   sola probabilidad P(arriba en 24h), con actualización de Bayes sobre frecuencias históricas.
4. **Validación**: mide el porcentaje de aciertos direccionales del bias sobre el propio histórico.
5. Todo lo anterior se conserva: ciclos de Fourier, periodicidad intradía, filtrado de ruido,
   cambios de régimen, y la ruta hora a hora con niveles alcanzables.

**Uso:** `Entorno de ejecución → Ejecutar todas`. Ajusta `CONFIG` y `TARGET` en la celda 2.

> ⚠️ Distribuciones de probabilidad, no certezas. La banda del 80% falla 1 de cada 5 veces por
> diseño y las noticias no se modelan. La volatilidad es más pronosticable que la dirección:
> úsala para tamaño de posición y stops, no para adivinar el rumbo.


In [ ]:
# @title 1) Instalar dependencias
%pip install -q -U hmmlearn yfinance curl_cffi scipy arch
print("Dependencias listas.")


In [ ]:
# @title 2) Configuración
CONFIG = {
    "n_states":         3,       # regímenes del HMM de precio
    "n_paths":          10000,   # simulaciones Monte Carlo
    "horizon_hours":    48,      # horizonte del pronóstico (velas H1)
    "checkpoint_every": 4,       # puntos de control de la ruta
    "confidence":       0.80,    # banda de confianza
    "ou_kappa":         0.03,    # reversión a la media en régimen rango
    "seed":             42,
    # ---- Fourier
    "fft_fit_bars":     2048,
    "fft_min_period":   8.0,
    "fft_n_cycles":     6,
    "use_cycle_drift":  True,
    # ---- volatilidad
    "use_garch":        True,    # escalar la vol simulada con el pronóstico GARCH
    "use_hour_factor":  True,    # escalar la vol simulada con el perfil intradía
    "stop_k":           1.5,     # stop sugerido = k * vol diaria pronosticada
}

TARGET = None           # precio objetivo opcional (ej. 4280.0) o None
DATA_SOURCE = "yahoo"   # "yahoo" o "csv"
YAHOO_PERIOD = "1y"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as sps
print("Configuración cargada.")


In [ ]:
# @title 3) Datos H1 de XAUUSD (Yahoo con reintentos, o CSV de MT5)
import time

MIN_BARS = 1000


def _yahoo_api_directa(ticker, interval="1h", range_="1y"):
    import requests
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
    r = requests.get(url, params={"interval": interval, "range": range_},
                     headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"},
                     timeout=30)
    r.raise_for_status()
    res = r.json()["chart"]["result"][0]
    ts = pd.to_datetime(res["timestamp"], unit="s", utc=True)
    cl = res["indicators"]["quote"][0]["close"]
    return pd.Series(cl, index=ts, dtype="float64").dropna()


def load_yahoo(period="1y"):
    import yfinance as yf
    tickers = ["XAUUSD=X", "GC=F"]
    periods = [period, "6mo", "3mo"]
    last_err = None
    for attempt in range(1, 4):
        for ticker in tickers:
            for per in periods:
                try:
                    df = yf.download(ticker, interval="1h", period=per,
                                     progress=False, auto_adjust=True)
                    if df is not None and len(df) > MIN_BARS:
                        s = pd.to_numeric(df["Close"].squeeze(), errors="coerce").dropna()
                        print(f"[DATOS] {len(s)} velas H1 de {ticker} (yfinance, {per})")
                        return s
                except Exception as e:
                    last_err = e
                    print(f"[DATOS] yfinance falló con {ticker} {per}: {e}")
                try:
                    s = _yahoo_api_directa(ticker, "1h", per)
                    if len(s) > MIN_BARS:
                        print(f"[DATOS] {len(s)} velas H1 de {ticker} (API directa, {per})")
                        return s
                except Exception as e:
                    last_err = e
                    print(f"[DATOS] API directa falló con {ticker} {per}: {e}")
        if attempt < 3:
            wait = 30 * attempt
            print(f"[DATOS] Yahoo está limitando; espero {wait}s y reintento...")
            time.sleep(wait)
    raise RuntimeError(
        f"No pude descargar datos. Último error: {last_err}\n"
        "Espera unos minutos, o 'Desconectar y eliminar entorno' (cambia la IP), "
        "o usa DATA_SOURCE='csv'.")


def load_mt5_csv_upload():
    from google.colab import files
    print("Sube tu CSV exportado de MT5 (XAUUSD H1)...")
    uploaded = files.upload()
    path = list(uploaded.keys())[0]
    for sep in ["\t", ",", ";"]:
        try:
            df = pd.read_csv(path, sep=sep)
            if df.shape[1] >= 4:
                break
        except Exception:
            continue
    else:
        raise ValueError("No pude leer el CSV. Verifica el separador.")
    df.columns = [c.strip().strip("<>").upper() for c in df.columns]
    if "CLOSE" not in df.columns:
        raise ValueError(f"No encuentro la columna CLOSE. Columnas: {list(df.columns)}")
    closes = pd.to_numeric(df["CLOSE"], errors="coerce")
    idx = None
    if "DATE" in df.columns and "TIME" in df.columns:
        idx = pd.to_datetime(df["DATE"].astype(str) + " " + df["TIME"].astype(str),
                             errors="coerce")
    elif "DATE" in df.columns:
        idx = pd.to_datetime(df["DATE"].astype(str), errors="coerce")
    if idx is not None and idx.notna().mean() > 0.9:
        s = pd.Series(closes.values, index=idx).dropna()
    else:
        s = closes.dropna()
        print("[DATOS] CSV sin fecha/hora: se omite el análisis intradía")
    print(f"[DATOS] {len(s)} velas cargadas de {path}")
    return s


series = load_yahoo(YAHOO_PERIOD) if DATA_SOURCE == "yahoo" else load_mt5_csv_upload()
closes = series.values.astype(float)
returns = np.diff(np.log(closes))
has_time = isinstance(series.index, pd.DatetimeIndex)
print(f"[DATOS] Último cierre: {closes[-1]:.2f}")
if has_time:
    print(f"[DATOS] Rango: {series.index[0]} → {series.index[-1]}")


In [ ]:
# @title 4) FOURIER: ciclos dominantes de precio y volatilidad
def espectro_welch(x, nper_max=1024):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    nper = int(min(nper_max, 2 * (len(x) // 4)))
    freqs, psd = sps.welch(x, fs=1.0, nperseg=nper)
    return freqs[1:], psd[1:]


def ciclos_dominantes(freqs, psd, n=8, min_period=3.0):
    peaks, _ = sps.find_peaks(psd)
    periodos = 1.0 / freqs[peaks]
    ok = periodos >= min_period
    peaks, periodos = peaks[ok], periodos[ok]
    orden = np.argsort(psd[peaks])[::-1][:n]
    total = psd.sum()
    return [(periodos[i], 100.0 * psd[peaks[i]] / total) for i in orden]


f_r, p_r = espectro_welch(returns)
f_v, p_v = espectro_welch(np.abs(returns))
print("===== CICLOS DOMINANTES =====")
print("  PRECIO (retornos):")
for per, pot in ciclos_dominantes(f_r, p_r, n=6):
    print(f"    {per:7.1f} h (~{per/24:5.2f} d) | {pot:5.2f}% de la potencia")
print("  VOLATILIDAD (|retornos|):")
for per, pot in ciclos_dominantes(f_v, p_v, n=4):
    print(f"    {per:7.1f} h (~{per/24:5.2f} d) | {pot:5.2f}% de la potencia")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (f, p, tit) in zip(axes, [(f_r, p_r, "Espectro del precio"),
                                  (f_v, p_v, "Espectro de la volatilidad")]):
    ax.semilogx(1.0 / f, p, lw=1.2, color="#1565c0")
    for h in [24, 12, 8]:
        ax.axvline(h, color="#c62828", ls=":", lw=0.8)
    ax.set_xlabel("período (horas)"); ax.set_title(tit)
fig.tight_layout(); plt.show()


In [ ]:
# @title 5) Periodicidad intradía → factores horarios para el Monte Carlo
factor_by_hour = {}
if has_time:
    idx_ret = series.index[1:]
    df_h = pd.DataFrame({"hora": idx_ret.hour, "ret": returns})
    perfil = df_h.groupby("hora")["ret"].agg(ret_medio="mean", vol="std", n="count")
    fac = (perfil["vol"] / perfil["vol"].mean()).clip(0.6, 1.8)
    factor_by_hour = fac.to_dict()

    top = perfil["vol"].sort_values(ascending=False)
    print("===== PERIODICIDAD INTRADÍA (UTC) =====")
    print("  Horas más volátiles:  " +
          ", ".join(f"{h:02d}:00 ({v*100:.3f}%)" for h, v in top.head(4).items()))
    print("  Horas más tranquilas: " +
          ", ".join(f"{h:02d}:00" for h in perfil["vol"].sort_values().head(3).index))
    print("  → Estos factores escalan la volatilidad simulada hora a hora en el MC.")

    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.bar(fac.index, fac.values, color="#1565c0", alpha=0.85)
    ax.axhline(1.0, color="k", lw=0.8, ls=":")
    ax.set_title("Factor de volatilidad por hora del día (1.0 = promedio)")
    ax.set_xlabel("hora UTC"); ax.set_ylabel("factor")
    fig.tight_layout(); plt.show()
else:
    print("[INTRADÍA] Sin marca de tiempo: factores horarios desactivados.")


In [ ]:
# @title 6) FOURIER: filtrado de ruido y dirección ARRIBA / ABAJO / LATERAL
N_FIT  = int(min(CONFIG["fft_fit_bars"], len(closes)))
H      = CONFIG["horizon_hours"]
x      = np.log(closes[-N_FIT:])
t      = np.arange(N_FIT)

coef    = np.polyfit(t, x, 1)
detrend = x - np.polyval(coef, t)
F   = np.fft.rfft(detrend)
fr  = np.fft.rfftfreq(N_FIT, d=1.0)
amp = np.abs(F) / N_FIT
valido = (fr > 0) & (1.0 / np.maximum(fr, 1e-12) >= CONFIG["fft_min_period"]) \
                  & (1.0 / np.maximum(fr, 1e-12) <= N_FIT / 2)
orden = [i for i in np.argsort(amp)[::-1] if valido[i]][:CONFIG["fft_n_cycles"]]

t_ext = np.arange(N_FIT + H)
recon = np.polyval(coef, t_ext)
for i in orden:
    recon += 2.0 * amp[i] * np.cos(2 * np.pi * fr[i] * t_ext + np.angle(F[i]))
suave, proyeccion = recon[:N_FIT], recon[N_FIT:]

delta   = proyeccion[-1] - recon[N_FIT - 1]
sigma_h = np.std(returns[-500:]) * np.sqrt(H)
ratio   = delta / sigma_h
veredicto = "ARRIBA" if ratio > 0.5 else ("ABAJO" if ratio < -0.5 else "LATERAL")
print(f"===== VEREDICTO FOURIER A {H}h =====")
print(f"  Cambio proyectado: {delta*100:+.2f}% | vol del horizonte: {sigma_h*100:.2f}% "
      f"| señal/ruido: {ratio:+.2f} → {veredicto}")

cycle_drift = np.diff(recon[N_FIT - 1:]) if CONFIG["use_cycle_drift"] else None

VER = 400
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(t[-VER:], np.exp(x[-VER:]), lw=0.8, color="#9e9e9e", label="precio real")
ax.plot(t[-VER:], np.exp(suave[-VER:]), lw=2, color="#1565c0", label="ciclos filtrados")
ax.plot(np.arange(N_FIT, N_FIT + H), np.exp(proyeccion), lw=2, ls="--", color="#c62828",
        label=f"extrapolación {H}h")
ax.axvline(N_FIT - 1, color="k", ls=":", lw=1)
ax.set_title(f"Filtrado de ruido por Fourier → {veredicto}")
ax.legend(loc="best"); fig.tight_layout(); plt.show()


In [ ]:
# @title 7) MARKOV bayesiano: HMM de precio con probabilidad POSTERIOR de regímenes
def fit_hmm_precio(returns, n_states, seed):
    """Devuelve (trans, mus, sigmas, hidden, posterior_ultima_vela) ordenados:
       0=alcista, 1=bajista, 2=rango. El posterior es la clave bayesiana: cuánta
       probabilidad tiene cada régimen AHORA, en vez de una etiqueta dura."""
    from hmmlearn.hmm import GaussianHMM
    SCALE = 100.0
    drift = pd.Series(returns).rolling(24, min_periods=1).mean().values
    X = np.column_stack([returns * SCALE, drift * SCALE * 20.0])
    best_model, best_ll = None, -np.inf
    for s in range(5):
        m = GaussianHMM(n_components=n_states, covariance_type="diag",
                        n_iter=500, random_state=seed + s, tol=1e-6, min_covar=1e-6)
        m.fit(X)
        ll = m.score(X)
        if ll > best_ll:
            best_model, best_ll = m, ll
    model  = best_model
    trans  = model.transmat_
    mus    = model.means_[:, 0] / SCALE
    covs   = np.array([np.diag(c)[0] for c in model.covars_])
    sigmas = np.sqrt(covs) / SCALE
    order_key = model.means_[:, 1]
    hidden = model.predict(X)
    post   = model.predict_proba(X)[-1]
    order = [int(np.argmax(order_key)), int(np.argmin(order_key))]
    order.append([s for s in range(n_states) if s not in order][0])
    idx   = np.array(order)
    remap = {old: new for new, old in enumerate(idx)}
    return (trans[np.ix_(idx, idx)], mus[idx], sigmas[idx],
            np.array([remap[s] for s in hidden]), post[idx])


trans, mus, sigmas, hidden, post_precio = fit_hmm_precio(
    returns, CONFIG["n_states"], CONFIG["seed"])
nombres = ["ALCISTA", "BAJISTA", "RANGO"]
current_state = int(np.argmax(post_precio))

print("===== REGÍMENES DE PRECIO (Markov) =====")
for s in range(3):
    occup = 100.0 * np.mean(hidden == s)
    print(f"  {nombres[s]:8s}| mu={mus[s]*100:+.4f}%/h  sigma={sigmas[s]*100:.4f}%  "
          f"ocupación={occup:5.1f}%")
print("\n  PROBABILIDAD POSTERIOR AHORA (arranque bayesiano del MC):")
for s in range(3):
    barra = "█" * int(round(post_precio[s] * 30))
    print(f"    {nombres[s]:8s}| {post_precio[s]*100:5.1f}% {barra}")
print(f"  Régimen más probable: {nombres[current_state]}"
      f" — pero los 10,000 caminos arrancan repartidos según estas probabilidades.")


In [ ]:
# @title 8) VOLATILIDAD: pronóstico GARCH(1,1) + regímenes CALMA / NORMAL / TORMENTA
from arch import arch_model

# --- GARCH(1,1): pronostica la volatilidad de cada una de las próximas H horas
am  = arch_model(returns * 100, vol="GARCH", p=1, q=1, mean="Zero", dist="t",
                 rescale=False)
res = am.fit(disp="off")
fore = res.forecast(horizon=H, reindex=False)
sigma_fore = np.sqrt(fore.variance.values[0]) / 100.0   # sigma por hora futura
sigma_hist = float(returns.std())
cond_vol   = np.asarray(res.conditional_volatility)
sigma_now  = float(cond_vol[-1]) / 100.0

p0 = float(closes[-1])
sigma_dia  = sigma_fore[:min(24, H)]
rango_dia  = 1.6 * p0 * float(np.sqrt((sigma_dia ** 2).sum()))   # E[max-min] aprox. browniano
stop_usd   = CONFIG["stop_k"] * p0 * float(np.sqrt((sigma_dia ** 2).sum()))

print("===== PRONÓSTICO DE VOLATILIDAD (GARCH) =====")
print(f"  Vol condicional AHORA: {sigma_now*100:.3f}%/h "
      f"(promedio histórico {sigma_hist*100:.3f}%/h → "
      f"{'ELEVADA' if sigma_now > 1.2*sigma_hist else 'BAJA' if sigma_now < 0.8*sigma_hist else 'NORMAL'})")
print(f"  Rango esperado próximas 24h: ±{rango_dia/2:.2f} USD (≈ {rango_dia:.2f} de alto a bajo)")
print(f"  Stop sugerido por volatilidad (k={CONFIG['stop_k']}): {stop_usd:.2f} USD "
      f"({stop_usd/0.10:.0f} pips)")

# --- HMM de volatilidad: CALMA / NORMAL / TORMENTA sobre la vol suavizada
from hmmlearn.hmm import GaussianHMM
ewma_sig = np.sqrt(pd.Series(returns ** 2).ewm(span=12).mean().values)
lv = np.log(ewma_sig + 1e-8).reshape(-1, 1)
hv = GaussianHMM(n_components=3, covariance_type="diag", n_iter=300,
                 random_state=CONFIG["seed"], min_covar=1e-6)
hv.fit(lv)
orden_v = np.argsort(hv.means_.ravel())          # 0=calma, 1=normal, 2=tormenta
remap_v = {old: new for new, old in enumerate(orden_v)}
vstates = np.array([remap_v[s] for s in hv.predict(lv)])
post_v  = hv.predict_proba(lv)[-1][orden_v]
nombres_v = ["CALMA", "NORMAL", "TORMENTA"]
vol_state = int(np.argmax(post_v))

print("\n===== RÉGIMEN DE VOLATILIDAD (Markov sobre la vol) =====")
for s in range(3):
    occup = 100.0 * np.mean(vstates == s)
    vol_media = float(np.exp(hv.means_.ravel()[orden_v][s]))
    print(f"  {nombres_v[s]:9s}| vol típica {vol_media*100:.3f}%/h | "
          f"ocupación {occup:5.1f}% | prob. ahora {post_v[s]*100:5.1f}%")
print(f"\n  >>> RÉGIMEN DE VOLATILIDAD ACTUAL: {nombres_v[vol_state]} <<<")
if vol_state == 0:
    print("  (compresión: los estallidos suelen nacer de aquí — cuidado con rupturas)")
elif vol_state == 2:
    print("  (tormenta: bandas anchas, reduce tamaño de posición)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(cond_vol, lw=0.8, color="#1565c0")
axes[0].set_title("Volatilidad condicional GARCH (histórico, %/h)")
axes[0].set_xlabel("velas H1")
axes[1].plot(np.arange(1, H + 1), sigma_fore * 100, lw=2, color="#c62828")
axes[1].axhline(sigma_hist * 100, color="k", ls=":", lw=1, label="promedio histórico")
axes[1].set_title(f"Pronóstico GARCH próximas {H}h (%/h)")
axes[1].set_xlabel("horas"); axes[1].legend()
fig.tight_layout(); plt.show()


In [ ]:
# @title 9) BIAS BAYESIANO: P(arriba en 24h) combinando las evidencias + validación
LOOK = 24   # horizonte del bias

# --- evidencias históricas por vela (alineadas con returns, largo n-1)
n = len(returns)
mom24 = np.zeros(n, dtype=int)          # momentum: ¿las últimas 24h subieron?
logc = np.log(closes)
for t_ in range(n):
    ini = max(0, t_ - LOOK + 1)
    mom24[t_] = 1 if logc[t_ + 1] - logc[ini] > 0 else 0

if has_time:
    horas_all = series.index[1:].hour.values
    ses = np.where((horas_all >= 6) & (horas_all < 12), 1,
          np.where((horas_all >= 12) & (horas_all < 21), 2, 0))   # 0=Asia 1=Londres 2=NY
else:
    ses = np.zeros(n, dtype=int)

# etiqueta: ¿el precio sube en las PRÓXIMAS 24h?
valid_t = n - LOOK
y = (logc[1 + LOOK: 1 + LOOK + valid_t] - logc[1: 1 + valid_t]) > 0

feats = {
    "régimen precio": hidden[:valid_t],
    "régimen vol":    vstates[:valid_t],
    "momentum 24h":   mom24[:valid_t],
    "sesión":         ses[:valid_t],
}

def likelihood_ratios(f, y, k):
    """P(valor|sube)/P(valor|baja) con suavizado de Laplace."""
    lrs = {}
    for v in range(k):
        pu = (np.sum((f == v) &  y) + 1.0) / (np.sum(y)  + k)
        pd_ = (np.sum((f == v) & ~y) + 1.0) / (np.sum(~y) + k)
        lrs[v] = pu / pd_
    return lrs

n_vals = {"régimen precio": 3, "régimen vol": 3, "momentum 24h": 2, "sesión": 3}
tablas = {nombre: likelihood_ratios(f, y, n_vals[nombre]) for nombre, f in feats.items()}

# --- estado ACTUAL de cada evidencia
actual = {
    "régimen precio": current_state,
    "régimen vol":    vol_state,
    "momentum 24h":   int(logc[-1] - logc[-min(LOOK, len(logc)-1)] > 0),
    "sesión":         int(ses[-1]) if has_time else 0,
}
etiquetas = {
    "régimen precio": nombres, "régimen vol": nombres_v,
    "momentum 24h": ["bajista", "alcista"], "sesión": ["Asia", "Londres", "NY"],
}

odds = 1.0   # prior: P(arriba)=50%
print("===== BIAS BAYESIANO A 24h =====")
print(f"  {'evidencia':16s} | {'estado actual':12s} | razón de verosimilitud")
for nombre in feats:
    v  = actual[nombre]
    lr = tablas[nombre][v]
    odds *= lr
    print(f"  {nombre:16s} | {etiquetas[nombre][v]:12s} | x{lr:.3f} "
          f"{'(empuja ARRIBA)' if lr > 1.02 else '(empuja ABAJO)' if lr < 0.98 else '(neutral)'}")
p_up = odds / (1.0 + odds)
bias = "ALCISTA" if p_up > 0.55 else ("BAJISTA" if p_up < 0.45 else "NEUTRO")
print(f"\n  >>> P(ARRIBA en 24h) = {p_up*100:.1f}%  →  BIAS {bias} <<<")

# --- validación sobre el histórico (¿acierta más que una moneda?)
lr_all = np.ones(valid_t)
for nombre, f in feats.items():
    tab = tablas[nombre]
    lr_all *= np.vectorize(tab.get)(f)
p_all   = lr_all / (1.0 + lr_all)
pred    = p_all > 0.5
acierto = float(np.mean(pred == y))
base    = float(max(np.mean(y), 1 - np.mean(y)))
print(f"\n===== VALIDACIÓN (mismo histórico, {valid_t} velas) =====")
print(f"  Acierto direccional del bias: {acierto*100:.1f}%")
print(f"  Referencia (siempre decir el lado más frecuente): {base*100:.1f}%")
print(f"  {'→ el bias AGREGA información' if acierto > base + 0.01 else '→ el bias apenas iguala la referencia: úsalo con escepticismo'}")
print("  Nota: validación en muestra (los mismos datos calibran y evalúan);")
print("  un resultado apenas superior a la referencia NO garantiza ventaja real.")


In [ ]:
# @title 10) MONTE CARLO v2: arranque posterior + volatilidad dinámica (GARCH × hora) + ciclos
def simulate_v2(trans, mus, sigmas, p0, start_probs, n_paths, n_bars, ou_kappa, seed,
                cycle_drift=None, vol_scale=None, hour_fac=None):
    rng    = np.random.default_rng(seed)
    states = rng.choice(len(start_probs), size=n_paths, p=start_probs)  # bayesiano
    logp   = np.full(n_paths, np.log(p0))
    anchor = logp.copy()
    cum_trans = np.cumsum(trans, axis=1)
    prices = np.empty((n_bars + 1, n_paths))
    prices[0] = p0
    for t_ in range(1, n_bars + 1):
        u = rng.random(n_paths)
        new_states = np.empty(n_paths, dtype=int)
        for s in range(3):
            mask = states == s
            if mask.any():
                new_states[mask] = np.searchsorted(cum_trans[s], u[mask])
        entrando = (new_states == 2) & (states != 2)
        anchor[entrando] = logp[entrando]
        states = new_states
        z  = rng.standard_normal(n_paths)
        cd = cycle_drift[t_ - 1] if cycle_drift is not None else 0.0
        esc = 1.0
        if vol_scale is not None:
            esc *= vol_scale[t_ - 1]
        if hour_fac is not None:
            esc *= hour_fac[t_ - 1]
        trend = states != 2
        logp = np.where(
            trend,
            logp + mus[np.clip(states, 0, 1)] + sigmas[np.clip(states, 0, 1)] * esc * z + cd,
            logp + ou_kappa * (anchor - logp) + sigmas[2] * esc * z + cd,
        )
        prices[t_] = np.exp(logp)
    return prices


vol_scale = np.clip(sigma_fore / sigma_hist, 0.5, 2.5) if CONFIG["use_garch"] else None

if CONFIG["use_hour_factor"] and has_time and factor_by_hour:
    fut_h = [(series.index[-1] + pd.Timedelta(hours=h)).hour for h in range(1, H + 1)]
    hour_fac = np.array([factor_by_hour.get(h, 1.0) for h in fut_h])
else:
    hour_fac = None

print(f"[MC] {CONFIG['n_paths']} caminos x {H}h desde {p0:.2f}")
print(f"     arranque posterior: " +
      " / ".join(f"{nombres[s]} {post_precio[s]*100:.0f}%" for s in range(3)))
print(f"     vol dinámica: GARCH {'sí' if vol_scale is not None else 'no'} | "
      f"factor horario {'sí' if hour_fac is not None else 'no'} | "
      f"drift de ciclos {'sí' if cycle_drift is not None else 'no'}")
paths = simulate_v2(trans, mus, sigmas, p0, post_precio,
                    CONFIG["n_paths"], H, CONFIG["ou_kappa"], CONFIG["seed"],
                    cycle_drift, vol_scale, hour_fac)
print("[MC] Listo.")


In [ ]:
# @title 11) RUTA hora a hora, niveles y objetivo
def report_route(paths, p0, cfg, target=None):
    conf   = cfg["confidence"]
    q_lo   = (1.0 - conf) / 2.0 * 100.0
    q_hi   = 100.0 - q_lo
    step   = cfg["checkpoint_every"]
    n_bars = paths.shape[0] - 1

    print(f"===== RUTA PRONOSTICADA ({paths.shape[1]} simulaciones, banda {conf*100:.0f}%) =====")
    print(f"  Inicio: XAUUSD {p0:.2f}\n")
    print(f"  {'hora':>6} | {'mediana':>9} | {'banda ' + format(conf*100,'.0f') + '%':^23} | {'P(subida)':>9}")
    print(f"  {'-'*6} | {'-'*9} | {'-'*23} | {'-'*9}")
    for h in range(step, n_bars + 1, step):
        med = np.median(paths[h])
        lo  = np.percentile(paths[h], q_lo)
        hi  = np.percentile(paths[h], q_hi)
        pup = 100.0 * np.mean(paths[h] > p0)
        print(f"  En {h:2d}h | {med:9.2f} | [{lo:9.2f} - {hi:9.2f}] | {pup:7.1f}%")

    run_max, run_min = paths.max(axis=0), paths.min(axis=0)
    lvl_up   = np.percentile(run_max, (1.0 - conf) * 100.0)
    lvl_down = np.percentile(run_min, conf * 100.0)
    print(f"\n===== NIVELES ALCANZABLES EN {n_bars}h =====")
    print(f"  Con {conf*100:.0f}% de prob. TOCA >= {lvl_up:.2f} ({(lvl_up-p0)/0.10:+.0f} pips)")
    print(f"  Con {conf*100:.0f}% de prob. TOCA <= {lvl_down:.2f} ({(lvl_down-p0)/0.10:+.0f} pips)")
    print(f"  Con 50% de prob. TOCA >= {np.percentile(run_max, 50):.2f} "
          f"| TOCA <= {np.percentile(run_min, 50):.2f}")

    if target is not None:
        if target > p0:
            p_touch = 100.0 * np.mean(run_max >= target)
            touch_h = [np.argmax(paths[:, j] >= target)
                       for j in range(paths.shape[1]) if run_max[j] >= target]
        else:
            p_touch = 100.0 * np.mean(run_min <= target)
            touch_h = [np.argmax(paths[:, j] <= target)
                       for j in range(paths.shape[1]) if run_min[j] <= target]
        print(f"\n===== OBJETIVO {target:.2f} =====")
        print(f"  Probabilidad de TOCARLO en {n_bars}h: {p_touch:.1f}%")
        if touch_h:
            print(f"  Hora típica del toque: {int(np.median(touch_h))}h "
                  f"| p10 {int(np.percentile(touch_h,10))}h | p90 {int(np.percentile(touch_h,90))}h")


report_route(paths, p0, CONFIG, TARGET)


In [ ]:
# @title 12) Gráficos y SÍNTESIS FINAL v2
n_bars = paths.shape[0] - 1
hours  = np.arange(n_bars + 1)

fig, ax = plt.subplots(figsize=(12, 5.5))
for lo_, hi_, a in [(2.5, 97.5, 0.15), (10, 90, 0.25), (25, 75, 0.35)]:
    ax.fill_between(hours, np.percentile(paths, lo_, axis=1),
                    np.percentile(paths, hi_, axis=1),
                    color="#1565c0", alpha=a, label=f"banda {hi_-lo_:.0f}%")
ax.plot(hours, np.median(paths, axis=1), "k-", lw=2, label="mediana")
ax.plot(np.arange(1, n_bars + 1), np.exp(proyeccion[:n_bars]), ls="--", lw=1.6,
        color="#ef6c00", label="proyección Fourier")
rng_ = np.random.default_rng(1)
for j in rng_.choice(paths.shape[1], size=25, replace=False):
    ax.plot(hours, paths[:, j], lw=0.4, alpha=0.4, color="#616161")
ax.axhline(p0, color="k", ls=":", lw=1)
if TARGET is not None:
    ax.axhline(TARGET, color="#c62828", ls="--", lw=1.5, label=f"objetivo {TARGET:.2f}")
ax.set_title(f"XAUUSD a {n_bars}h — MC bayesiano con volatilidad dinámica "
             f"(nota: las bandas respiran con las sesiones)")
ax.set_xlabel("horas"); ax.set_ylabel("USD/oz"); ax.legend(loc="upper left")
fig.tight_layout(); fig.savefig("xauusd_v2_fanchart.png", dpi=120); plt.show()

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.hist(paths[-1], bins=120, color="#2e7d32", alpha=0.85)
ax.axvline(p0, color="k", ls=":", lw=1.5, label=f"inicio {p0:.2f}")
ax.axvline(np.median(paths[-1]), color="#c62828", lw=1.5,
           label=f"mediana {np.median(paths[-1]):.2f}")
ax.set_title(f"Distribución del precio a {n_bars}h")
ax.legend(); fig.tight_layout()
fig.savefig("xauusd_v2_final.png", dpi=120); plt.show()

med_fin = float(np.median(paths[-1]))
pup_fin = 100.0 * np.mean(paths[-1] > p0)
mc_dir  = "ARRIBA" if pup_fin > 55 else ("ABAJO" if pup_fin < 45 else "LATERAL")

print("=" * 64)
print("                     SÍNTESIS FINAL v2")
print("=" * 64)
print(f"  DIRECCIÓN")
print(f"   · Fourier (ciclos):        {veredicto} (señal/ruido {ratio:+.2f})")
print(f"   · Bias bayesiano 24h:      {bias} (P(arriba) {p_up*100:.1f}%)")
print(f"   · Monte Carlo {n_bars}h:        {mc_dir} (P(subida) {pup_fin:.1f}%, "
      f"mediana {med_fin:.2f}, {(med_fin/p0-1)*100:+.2f}%)")
print(f"  RÉGIMEN")
print(f"   · Precio (posterior):      " +
      " / ".join(f"{nombres[s]} {post_precio[s]*100:.0f}%" for s in range(3)))
print(f"   · Volatilidad:             {nombres_v[vol_state]} "
      f"(vol ahora {sigma_now*100:.3f}%/h vs {sigma_hist*100:.3f}%/h histórica)")
print(f"  RIESGO (esto es lo más confiable del reporte)")
print(f"   · Rango esperado 24h:      ≈ {rango_dia:.2f} USD de alto a bajo")
print(f"   · Stop sugerido por vol:   {stop_usd:.2f} USD ({stop_usd/0.10:.0f} pips)")
print(f"   · Validación del bias:     {acierto*100:.1f}% de acierto "
      f"(referencia {base*100:.1f}%)")
print("=" * 64)
print("  La dirección es la parte débil de cualquier modelo; la volatilidad,")
print("  la fuerte. Dimensiona la posición con el rango y el stop de arriba.")
